# 公開合成素材版v3｜Colabでの再現
作成：2026/10/26 14:07 JST／検証状況更新：2026/10/26 18:14 JST

2026/10/26のユーザーColab実測では、公開合成素材の5ケースについて等級判定・把持・搬送が5/5成功。元ログの `all_pass:false` と終了コード2は未実行の10ケース総合条件も含むためであり、未実行条件を合格扱いにはしない。

5/5は開発に使用した合成条件での成功であり、独立の実果精度ではない。安全ゲート0.76・既存の物理制御は維持。

セルを上から実行し、この公開リポジトリのZIPを1個アップロードする。失敗した場合もログを保存する。


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess, json, hashlib, shutil
# Upload widget only works after running this cell in the current browser session.
up=files.upload()
assert len(up)==1 and next(iter(up)).lower().endswith('.zip'), '新しいv3 ZIPを1個アップロードする'
name=next(iter(up))
root=Path('/content/physical_ai_public_v3')
if root.exists(): shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(name) as z:
 assert z.testzip() is None, 'ZIP CRC error'
 for item in z.infolist():
  p=Path(item.filename)
  assert not p.is_absolute() and '..' not in p.parts, p
 z.extractall(root)
repo=root if (root/'stage88').is_dir() else next(x for x in root.iterdir() if (x/'stage88').is_dir())
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'stage88/requirements.txt')],check=True)
font=Path('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
if not font.exists():
 subprocess.run(['apt-get','update','-qq'],check=True)
 subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=True)
print('V3_SYNTHETIC_PUBLIC_READY',repo)


In [ ]:
import subprocess,sys,os,json
from pathlib import Path
src=repo/'stage88'
chk=subprocess.run([sys.executable,str(repo/'scripts/check_public_completeness.py')],cwd=repo,text=True,capture_output=True)
print('PUBLIC_ASSETS_CHECK',chk.returncode,chk.stdout[-1000:],chk.stderr[-500:]); assert chk.returncode==0
pre=subprocess.run([sys.executable,'run_stage35_motion.py','--train-only'],cwd=src,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print('TRAIN_RC',pre.returncode,pre.stdout[-2000:]);assert pre.returncode==0
# Shared camera-model preflight uses synthetic calibration and MUST keep threshold=.76.
preflight=subprocess.run([sys.executable,'test_stage67_preflight.py'],cwd=src,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print('CAMERA_PREFLIGHT_RC',preflight.returncode,preflight.stdout[-1400:]); assert preflight.returncode==0
out=Path('/content/PAI_PUBLIC_SYNTHETIC_RESULTS_V3')
if out.exists():
 import shutil
 shutil.rmtree(out)
env={**os.environ,'STAGE84_SMALL_SCAR_DEMO':'1'}
run=subprocess.run([sys.executable,'run_stage70_continuous.py','--output',str(out)],cwd=src,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print('PYBULLET_RC',run.returncode,run.stdout[-7500:])
sumf=out/'summary.json'
if sumf.exists():
 summ=json.loads(sumf.read_text());print('N_EPISODES',summ.get('n_episodes'),'ALL_PASS',summ.get('all_pass'))
 for rep in sorted((out/'case_reports').glob('*.json')):
  e=json.loads(rep.read_text());gate=e.get('stage32') or {}
  print('CASE',e.get('case'),'GATE',gate.get('allow_descent'),'PRED', (gate.get('object') or {}).get('predicted_object'),'CLASS',(e.get('classification') or {}).get('class'),'PHYSICS',e.get('physical_status'))
else: print('NO SUMMARY - see raw logs')
# Save evidence even if returncode == 2. A strict failure MUST NOT be concealed.


In [ ]:
from pathlib import Path
import zipfile
folder=Path('/content/PAI_PUBLIC_SYNTHETIC_RESULTS_V3')
pack=Path('/content/PAI_PUBLIC_SYNTHETIC_RESULT_V3.zip')
assert folder.exists(),'No experiment directory'
with zipfile.ZipFile(pack,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in sorted(folder.rglob('*')):
  if p.is_file() and p.suffix.lower() in ('.json','.csv','.mp4','.txt','.png'):
   z.write(p,p.relative_to(folder))
print('保存:',pack,pack.stat().st_size)
from google.colab import files
files.download(str(pack))
